# Fusion Strategy Comparison (mBRSET)

Compares the single-image baseline against mean pooling, max pooling, and transformer-based fusion across all available images per patient. 

In [ ]:
import sys, os, glob

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import torch
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score

from model import UnifiedBackbone, UnifiedBackboneMulti
from fundus_dataset import FundusDataset
from multi_image_dataset import MultiImageFundusDataset

## Configuration

In [ ]:
DATASET    = "mBRSET"
MODEL_NAME = "retfound_green"

MBRSET_DATA_DIR = r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\data"
MBRSET_IMG_ROOT = r"C:\Users\preet\Documents\mBRSET\mbrset-a-mobile-brazilian-retinal-dataset-1.0\images"

SINGLE_CKPT_PATTERN = "run4_mBRSET_single__img_diagnosis_model_top*.pth"
MULTI_CKPT_PATTERN  = "run2_mBRSET_multi__img_diagnosis_model_top*.pth"
N_CHECKPOINTS       = 5
NUM_IMAGES          = 4
IMG_INDICES_ALL     = [0, 1, 2, 3]

## Load Data

In [ ]:
test_df  = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_test_full.pkl"))
img_root = MBRSET_IMG_ROOT
test_df.dropna(subset=["final_icdr"], inplace=True)
print(f"Test rows: {len(test_df)}")

## Transforms

In [ ]:
mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
val_tf = A.Compose([
    A.Resize(392, 392),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

## Helper Functions

In [ ]:
@torch.no_grad()
def get_single_probs(model, img_idx, device):
    """Run single-image model on test set for one img_idx. Returns (probs, labels)."""
    ds = FundusDataset(test_df, img_root, high_quality_tf=val_tf, low_quality_tf=val_tf,
                       label_col="final_icdr", img_idx=img_idx)
    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=0)
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels, _ in tqdm(loader, desc=f"img_idx={img_idx}", leave=False):
        imgs = imgs.to(device)
        with autocast(dtype=torch.float16):
            logits = model(imgs)
        all_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_probs), np.array(all_labels)


@torch.no_grad()
def get_multi_probs(model, device, img_indices=None):
    """Run the transformer fusion model on the test set. Returns (probs, labels)."""
    ds = MultiImageFundusDataset(test_df, img_root, transform=val_tf, label_col="final_icdr",
                                 patient_col="patient", num_images=NUM_IMAGES, img_indices=img_indices)
    loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)
    model.eval()
    all_probs, all_labels = [], []
    for imgs, mask, labels, _ in tqdm(loader, desc="multi", leave=False):
        imgs, mask = imgs.to(device), mask.to(device)
        with autocast(dtype=torch.float16):
            logits = model(imgs, mask)
        all_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_probs), np.array(all_labels)


def compute_metrics(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tp = int(((preds == 1) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    return {
        "BA":          0.5 * (sensitivity + specificity),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "AUPRC":       average_precision_score(labels, probs),
        "AUROC":       roc_auc_score(labels, probs),
        "F1":          f1_score(labels, preds, zero_division=0),
        "PPV":         precision_score(labels, preds, zero_division=0),
    }

## Find Checkpoints

In [ ]:
device       = "cuda"
single_ckpts = sorted(glob.glob(SINGLE_CKPT_PATTERN))[:N_CHECKPOINTS]
multi_ckpts  = sorted(glob.glob(MULTI_CKPT_PATTERN))[:N_CHECKPOINTS]

assert single_ckpts, f"No checkpoints matched: {SINGLE_CKPT_PATTERN}"
assert multi_ckpts,  f"No checkpoints matched: {MULTI_CKPT_PATTERN}"
print(f"Single-image checkpoints: {len(single_ckpts)}  |  Multi-image checkpoints: {len(multi_ckpts)}")

## Single-Image Baseline

In [ ]:
single_results = []
for ckpt_path in single_ckpts:
    model = UnifiedBackbone(model_name=MODEL_NAME)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device).eval()
    for img_idx in IMG_INDICES_ALL:
        probs, labels = get_single_probs(model, img_idx, device)
        single_results.append(compute_metrics(probs, labels))

print(f"Single image: {len(single_results)} evaluations")

## Mean & Max Fusion

In [ ]:
mean_fusion_results, max_fusion_results = [], []
for ckpt_path in single_ckpts:
    model = UnifiedBackbone(model_name=MODEL_NAME)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device).eval()

    probs_per_idx = []
    for img_idx in IMG_INDICES_ALL:
        probs, labels = get_single_probs(model, img_idx, device)
        probs_per_idx.append(probs)
    probs_stack = np.stack(probs_per_idx, axis=0)

    mean_fusion_results.append(compute_metrics(probs_stack.mean(axis=0), labels))
    max_fusion_results.append(compute_metrics(probs_stack.max(axis=0), labels))

print(f"Mean/Max fusion: {len(mean_fusion_results)} evaluations each")

## Transformer Fusion (all images)

In [ ]:
multi_results = []
for ckpt_path in multi_ckpts:
    model = UnifiedBackboneMulti(model_name=MODEL_NAME, num_images=NUM_IMAGES)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device).eval()
    probs, labels = get_multi_probs(model, device)
    multi_results.append(compute_metrics(probs, labels))

print(f"Transformer fusion: {len(multi_results)} evaluations")

## Summary Table

In [ ]:
metrics_order = ["BA", "Sensitivity", "Specificity", "AUPRC", "AUROC", "F1", "PPV"]

def summarize(results_list):
    return {m: f"{np.mean([r[m] for r in results_list]):.3f} \u00b1 {np.std([r[m] for r in results_list]):.3f}"
            for m in metrics_order}

rows = {
    f"Single Image Baseline (n={len(single_results)})": summarize(single_results),
    f"Mean Fusion (n={len(mean_fusion_results)})":       summarize(mean_fusion_results),
    f"Max Fusion (n={len(max_fusion_results)})":         summarize(max_fusion_results),
    f"Transformer Fusion (n={len(multi_results)})":      summarize(multi_results),
}

df_table = pd.DataFrame(rows, index=metrics_order).T
df_table.index.name = "Model"

out_csv = f"{DATASET}_aggregation_comparison.csv"
df_table.to_csv(out_csv)
print(f"Saved: {out_csv}")
df_table